In [34]:
import geopandas as gpd
import pandas as pd

import duckdb


import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import contextily as cx

import folium

import numpy as np



In [20]:
## querry data via duckdb from parquets
# result is loaded to geodataframe

In [21]:
# Cell 1: Setup und Konfiguration


# Konfiguration
# DATE_SNAPSHOT = "2026-01-26"  # Für Titel und Dokumentation
HASHTAG_FILTER = '%#missing-cw_mapillary-signs%'
PARQUET_PATHS = [
    '~/ohsome-planet/data/germany_from2025_rep/contributions/history/way*-*-history-contribs.parquet',
    '~/ohsome-planet/data/germany_from2025_rep/contributions/latest/way*-*-latest-contribs.parquet',
    '~/ohsome-planet/data/germany_from2025_rep/updates/000/004/*.parquet'
    #\\ ohsome-planet\data\germany_from2025_rep\updates\000\004
]
  # '~/ohsome-planet/out-germany_cs_251201/contributions/history/way*-*-history-contribs.parquet',
  #   '~/ohsome-planet/out-germany_cs_251201/contributions/latest/way*-*-latest-contribs.parquet'

# Cell 2: Datenvorbereitung (einmalig, dauert 1min)
duckdb.sql("INSTALL spatial; LOAD spatial;")

duckdb.sql(f"""
CREATE OR REPLACE TABLE cycleway_base AS
SELECT
  geometry AS geom,
  tags,
  tags_before,
  "user" AS usr,
  changeset,
  osm_type,
  osm_id,
  osm_version,
  valid_from,
  contrib_type,
  length,
  length_delta
FROM read_parquet({PARQUET_PATHS}, union_by_name=true)
WHERE COALESCE(changeset.tags.hashtags,'') ILIKE '{HASHTAG_FILTER}';
""")

print("✅ Datenvorbereitung abgeschlossen")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Datenvorbereitung abgeschlossen


In [22]:
rows = duckdb.sql("SELECT ST_AsText(geom) AS wkt, tags, changeset, usr FROM cycleway_base").fetchall()
gdf = gpd.GeoDataFrame.from_records(rows, columns=["wkt", "tags", "changeset", "usr"])
gdf

,wkt,tags,changeset,usr
0,"LINESTRING (11.6018703 48.1337196, 11.6019611 ...","{'traffic_sign': 'DE:244.1,1024-10,1022-12', '...","{'id': 171582735, 'created_at': 2025-09-07 14:...","{'id': 3644159, 'name': 'bicyclett'}"
1,"LINESTRING (11.6018703 48.1337196, 11.6018069 ...","{'name': 'Wolfgangstraße', 'bicycle_road': 'ye...","{'id': 171582735, 'created_at': 2025-09-07 14:...","{'id': 3644159, 'name': 'bicyclett'}"
2,"LINESTRING (11.5945784 48.1322799, 11.5946342 ...","{'lane_markings': 'no', 'sidewalk:left': 'sepa...","{'id': 171582735, 'created_at': 2025-09-07 14:...","{'id': 3644159, 'name': 'bicyclett'}"
3,"LINESTRING (10.8036675 49.6983374, 10.8036516 ...","{'surface': 'asphalt', 'ref': 'St 2263', 'side...","{'id': 175556141, 'created_at': 2025-12-05 17:...","{'id': 3644159, 'name': 'bicyclett'}"
4,"LINESTRING (10.1007807 52.8445143, 10.1006372 ...","{'lit': 'yes', 'cycleway:right': 'no', 'highwa...","{'id': 170571393, 'created_at': 2025-08-17 17:...","{'id': 3644159, 'name': 'bicyclett'}"
...,...,...,...,...
7095,"POLYGON ((9.1822273 54.1720082, 9.1822273 54.1...","{'rcn_ref': '*-*', 'network:type': 'node_netwo...","{'id': 179844645, 'created_at': 2026-03-14 10:...","{'id': 3644159, 'name': 'bicyclett'}"
7096,"POLYGON ((9.179567 54.1707077, 9.179567 54.172...",{'description': 'Nordhastedt | Bahnhofstraße <...,"{'id': 179844645, 'created_at': 2026-03-14 10:...","{'id': 3644159, 'name': 'bicyclett'}"
7097,"POLYGON ((9.090665 54.1770298, 9.090665 54.202...","{'ref': '2', 'network:short': 'NAH.SH', 'route...","{'id': 179845008, 'created_at': 2026-03-14 10:...","{'id': 3644159, 'name': 'bicyclett'}"
7098,"POLYGON ((9.0979188 54.1452492, 9.0979188 54.2...","{'colour': '#BE6416', 'via': 'Albersdorf, ZOB'...","{'id': 179844645, 'created_at': 2026-03-14 10:...","{'id': 3644159, 'name': 'bicyclett'}"


In [23]:
gdf.changeset

0       {'id': 171582735, 'created_at': 2025-09-07 14:...
1       {'id': 171582735, 'created_at': 2025-09-07 14:...
2       {'id': 171582735, 'created_at': 2025-09-07 14:...
3       {'id': 175556141, 'created_at': 2025-12-05 17:...
4       {'id': 170571393, 'created_at': 2025-08-17 17:...
                              ...                        
7095    {'id': 179844645, 'created_at': 2026-03-14 10:...
7096    {'id': 179844645, 'created_at': 2026-03-14 10:...
7097    {'id': 179845008, 'created_at': 2026-03-14 10:...
7098    {'id': 179844645, 'created_at': 2026-03-14 10:...
7099    {'id': 179844645, 'created_at': 2026-03-14 10:...
Name: changeset, Length: 7100, dtype: object

In [24]:
# Neuestes Datum finden und formatieren ===
gdf["created_at"] = gdf["changeset"].apply(
    lambda u: u.get("created_at") if isinstance(u, dict) else None
)

# Neuestes Datum ermitteln
latest_date = gdf["created_at"].max()
DATE_SNAPSHOT = latest_date.strftime("%Y-%m-%d") if latest_date else "Unknown"

print(f"✅ Neuestes Changeset-Datum: {DATE_SNAPSHOT}")
print(f"   Ältestes Datum: {gdf['created_at'].min().strftime('%Y-%m-%d')}")

✅ Neuestes Changeset-Datum: 2026-03-14
   Ältestes Datum: 2025-08-06


In [25]:
#gdf.to_file("cycleway_base_2026-03-15.geojson", driver="GeoJSON")

In [26]:
def get_infrastructure_transitions_strict_df():
    """Gemeinsame Datenbasis fuer Karte und Sankey."""
    return duckdb.sql("""
    WITH base AS (
      SELECT
        geom,
        tags, tags_before,
        length AS len_after_m,
        length - COALESCE(length_delta, 0) AS len_before_m,
        COALESCE(length_delta, 0) AS dlen_m,
        osm_id, osm_version, valid_from
      FROM cycleway_base
      WHERE osm_type='way'
        AND tags['highway'] IS NOT NULL
        AND length IS NOT NULL
    ),
    flags_raw AS (
      SELECT
        geom, len_after_m, len_before_m, dlen_m, tags, tags_before,
        osm_id, osm_version, valid_from,

        ((tags['highway']='path')    AND (tags['bicycle']='designated'))  AS after_path_desig_raw,
        ( tags['highway']='cycleway')                                     AS after_cycleway_raw,
        ((tags['highway']='footway') AND (tags['bicycle']='yes'))         AS after_footway_bike_raw,
        ((tags['highway']='track')   AND (tags['bicycle']='designated'))  AS after_track_bike_raw,
        ( tags['bicycle_road']='yes')                                     AS after_bicycle_road_raw,
        ( COALESCE(tags['highway'],'') <> 'cycleway'
          AND (tags['cycleway']='track' OR tags['cycleway:left']='track'
               OR tags['cycleway:right']='track' OR tags['cycleway:both']='track')
        ) AS after_track_raw,
        ( COALESCE(tags['highway'],'') <> 'cycleway'
          AND (tags['cycleway']='lane' OR tags['cycleway:left']='lane'
               OR tags['cycleway:right']='lane' OR tags['cycleway:both']='lane')
        ) AS after_lane_raw,

        ((tags_before['highway']='path')    AND (tags_before['bicycle']='designated'))  AS before_path_desig_raw,
        ( tags_before['highway']='cycleway')                                           AS before_cycleway_raw,
        ((tags_before['highway']='footway') AND (tags_before['bicycle']='yes'))        AS before_footway_bike_raw,
        ((tags_before['highway']='track')   AND (tags_before['bicycle']='designated')) AS before_track_bike_raw,
        ( tags_before['bicycle_road']='yes')                                           AS before_bicycle_road_raw,
        ( COALESCE(tags_before['highway'],'') <> 'cycleway'
          AND (tags_before['cycleway']='track' OR tags_before['cycleway:left']='track'
               OR tags_before['cycleway:right']='track' OR tags_before['cycleway:both']='track')
        ) AS before_track_raw,
        ( COALESCE(tags_before['highway'],'') <> 'cycleway'
          AND (tags_before['cycleway']='lane' OR tags_before['cycleway:left']='lane'
               OR tags_before['cycleway:right']='lane' OR tags_before['cycleway:both']='lane')
        ) AS before_lane_raw,

        tags_before['highway'] AS before_highway
      FROM base
    ),
    flags AS (
      SELECT
        geom, len_after_m, len_before_m, dlen_m, tags, tags_before, osm_id, osm_version, valid_from,
        COALESCE(after_path_desig_raw,FALSE)    AS after_path_desig,
        COALESCE(after_cycleway_raw,FALSE)      AS after_cycleway,
        COALESCE(after_footway_bike_raw,FALSE)  AS after_footway_bike,
        COALESCE(after_track_bike_raw,FALSE)    AS after_track_bike,
        COALESCE(after_bicycle_road_raw,FALSE)  AS after_bicycle_road,
        COALESCE(after_track_raw,FALSE)         AS after_track,
        COALESCE(after_lane_raw,FALSE)          AS after_lane,
        COALESCE(before_path_desig_raw,FALSE)   AS before_path_desig,
        COALESCE(before_cycleway_raw,FALSE)     AS before_cycleway,
        COALESCE(before_footway_bike_raw,FALSE) AS before_footway_bike,
        COALESCE(before_track_bike_raw,FALSE)   AS before_track_bike,
        COALESCE(before_bicycle_road_raw,FALSE) AS before_bicycle_road,
        COALESCE(before_track_raw,FALSE)        AS before_track,
        COALESCE(before_lane_raw,FALSE)         AS before_lane,
        before_highway
      FROM flags_raw
    ),
    transitions AS (
      SELECT
        geom, osm_id, osm_version, valid_from,
        CASE
          WHEN after_path_desig THEN 'hw=path_bicycle=designated'
          WHEN after_cycleway THEN 'hw=cycleway'
          WHEN after_footway_bike THEN 'hw=footway_bicycle=yes'
          WHEN after_track_bike THEN 'hw=track_bicycle=designated'
          WHEN after_bicycle_road THEN 'bicycle_road'
          WHEN after_track THEN 'cw=track'
          WHEN after_lane THEN 'cw=lane'
          ELSE NULL
        END AS target_category,
        CASE
          WHEN before_path_desig THEN 'hw=path_bicycle=designated'
          WHEN before_cycleway THEN 'hw=cycleway'
          WHEN before_footway_bike THEN 'hw=footway_bicycle=yes'
          WHEN before_track_bike THEN 'hw=track_bicycle=designated'
          WHEN before_bicycle_road THEN 'bicycle_road'
          WHEN before_track THEN 'cw=track'
          WHEN before_lane THEN 'cw=lane'
          WHEN before_highway IS NOT NULL THEN 'hw=' || before_highway || '_other'
          ELSE 'Added'
        END AS source_category,
        CASE
          WHEN after_path_desig AND NOT before_path_desig THEN len_after_m
          WHEN after_path_desig AND before_path_desig AND dlen_m > 0 THEN dlen_m
          WHEN after_cycleway AND NOT before_cycleway THEN len_after_m
          WHEN after_cycleway AND before_cycleway AND dlen_m > 0 THEN dlen_m
          WHEN after_footway_bike AND NOT before_footway_bike THEN len_after_m
          WHEN after_footway_bike AND before_footway_bike AND dlen_m > 0 THEN dlen_m
          WHEN after_track_bike AND NOT before_track_bike THEN len_after_m
          WHEN after_track_bike AND before_track_bike AND dlen_m > 0 THEN dlen_m
          WHEN after_bicycle_road AND NOT before_bicycle_road THEN len_after_m
          WHEN after_bicycle_road AND before_bicycle_road AND dlen_m > 0 THEN dlen_m
          WHEN after_track AND NOT before_track THEN len_after_m
          WHEN after_track AND before_track AND dlen_m > 0 THEN dlen_m
          WHEN after_lane AND NOT before_lane THEN len_after_m
          WHEN after_lane AND before_lane AND dlen_m > 0 THEN dlen_m
          ELSE 0
        END AS added_length_m
      FROM flags
      WHERE (
        after_path_desig OR after_cycleway OR after_footway_bike OR after_track_bike
        OR after_bicycle_road OR after_track OR after_lane
      )
    )
    SELECT
      ST_AsWKB(geom) AS geometry,
      source_category AS source,
      target_category AS target,
      added_length_m / 1000.0 AS length_km,
      osm_id, osm_version, valid_from
    FROM transitions
    WHERE target_category IS NOT NULL
      AND added_length_m > 0
      AND source_category != target_category;
    """).fetchdf()

In [27]:
# Funktion fuer Sankey-Diagramm Daten mit differenzierten Quellen
def get_infrastructure_analysis_sankey():
    """Aggregiert die gemeinsame Strict-Datenbasis fuer das Sankey-Diagramm."""
    transitions_df = get_infrastructure_transitions_strict_df()
    result = (
        transitions_df.groupby(['source', 'target'], as_index=False)['length_km']
        .sum()
        .rename(columns={'length_km': 'value'})
        .sort_values('value', ascending=False, ignore_index=True)
    )
    result['value'] = result['value'].round(1)
    return result

# Erstelle Sankey-Daten
sankey_df = get_infrastructure_analysis_sankey()
print("Sankey-Daten mit differenzierten Quellen:")
print(sankey_df)

Sankey-Daten mit differenzierten Quellen:
                         source                       target  value
0                         Added   hw=path_bicycle=designated  208.6
1                 hw=path_other   hw=path_bicycle=designated   82.7
2                hw=track_other   hw=path_bicycle=designated   35.7
3                   hw=cycleway   hw=path_bicycle=designated   35.5
4                hw=track_other  hw=track_bicycle=designated   31.5
5            hw=secondary_other                     cw=track   25.5
6              hw=footway_other   hw=path_bicycle=designated   24.4
7             hw=tertiary_other                     cw=track   22.2
8        hw=footway_bicycle=yes   hw=path_bicycle=designated   17.7
9                         Added       hw=footway_bicycle=yes   14.9
10           hw=secondary_other                      cw=lane   10.8
11  hw=track_bicycle=designated   hw=path_bicycle=designated    9.3
12             hw=footway_other       hw=footway_bicycle=yes    8.6
13    

In [28]:
# Sankey-Diagramm mit plotly erstellen (mit differenzierten Quellen)
import plotly.graph_objects as go
import numpy as np

# Erstelle Sankey-Daten
sankey_df = get_infrastructure_analysis_sankey()

# Filtere nur positive Werte
sankey_df = sankey_df[sankey_df['value'] > 0].copy()

# Hole alle eindeutigen Sources und Targets
sources = sorted(sankey_df['source'].unique())
targets = sorted(sankey_df['target'].unique())

# Berechne Gesamtwerte für Farbcodierung und Sortierung
source_totals = sankey_df.groupby('source')['value'].sum().to_dict()
target_totals = sankey_df.groupby('target')['value'].sum().to_dict()

# Sortiere nach Gesamtwert für bessere Positionierung
sources = sorted(sources, key=lambda x: source_totals.get(x, 0), reverse=True)
targets = sorted(targets, key=lambda x: target_totals.get(x, 0), reverse=True)

# Erstelle Liste aller Knoten: Sources links, Targets rechts (auch wenn sie überlappen)
# Jeder Knoten erscheint zweimal wenn er sowohl Source als auch Target ist
all_nodes = sources + targets
# Erstelle Indizes: Sources bekommen Indizes 0..len(sources)-1, Targets bekommen len(sources)..len(sources)+len(targets)-1
source_indices_map = {source: idx for idx, source in enumerate(sources)}
target_indices_map = {target: len(sources) + idx for idx, target in enumerate(targets)}

# Erstelle source, target und value Listen für plotly
source_indices = []
target_indices = []
values = []
link_colors = []

# Farben definieren
color_added = 'rgba(114, 56, 150, 0.5)'
color_other = 'rgba(67, 75, 163, 0.5)'

for _, row in sankey_df.iterrows():
    source = row['source']
    target = row['target']
    value = row['value']
    
    # Verwende separate Indizes für Sources (links) und Targets (rechts)
    source_indices.append(source_indices_map[source])
    target_indices.append(target_indices_map[target])
    values.append(value)
    
    # Farbe basierend auf Source
    if source == 'Added':
        link_colors.append(color_added)
    else:
        link_colors.append(color_other)

# Erstelle Knoten-Farben und Labels
node_colors = []
node_labels = []
node_x = []  # X-Positionen: 0 für Sources, 1 für Targets
node_y = []  # Y-Positionen: proportional zu Werten

# Berechne Y-Positionen für Sources (proportional zu Werten)
source_total = sum(source_totals.values())
y_current = 0
source_y_positions = {}
for source in sources:
    total = source_totals.get(source, 0)
    if total > 0:
        height = total / source_total if source_total > 0 else 0
        source_y_positions[source] = y_current + height / 2
        y_current += height

# Berechne Y-Positionen für Targets (proportional zu Werten)
target_total = sum(target_totals.values())
y_current = 0
target_y_positions = {}
for target in targets:
    total = target_totals.get(target, 0)
    if total > 0:
        height = total / target_total if target_total > 0 else 0
        target_y_positions[target] = y_current + height / 2 
        y_current += height


# # Berechne Y-Positionen für Targets (proportional zu Werten) + variable Gaps
# target_total = sum(target_totals.values())

# # Parameter zum Tuning
# min_height = 0.012      # Mindest-"Höhe" pro Node (macht kleine Nodes lesbarer)
# gap_base   = 0.002      # Grundabstand zwischen Nodes
# gap_scale  = 0.020      # wie stark kleine Nodes extra Abstand bekommen

# # 1) Roh-Heights
# heights = np.array([target_totals.get(t, 0) / target_total for t in targets], dtype=float)

# # 2) Mindesthöhe erzwingen (optional, aber sehr hilfreich bei vielen kleinen Nodes)
# heights = np.maximum(heights, min_height)

# # 3) Variable Gaps: kleine Nodes bekommen mehr Abstand
# #    (wenn height klein => (h_max - h) groß => gap groß)
# h_max = heights.max() if len(heights) else 0
# gaps = gap_base + gap_scale * (h_max - heights)

# # 4) Normalisieren: Sum(heights) + Sum(gaps) muss in [0..1] passen
# total_span = heights.sum() + gaps.sum()
# heights = heights / total_span
# gaps    = gaps    / total_span

# # 5) Midpoints berechnen
# target_y_positions = {}
# y_current = 0.0
# for t, h, g in zip(targets, heights, gaps):
#     target_y_positions[t] = y_current + h / 2
#     y_current += h + g

###############################
import numpy as np

def compute_y_positions(items, totals, min_height=0.012, gap_base=0.002, gap_scale=0.020):
    total_sum = sum(totals.values())
    if total_sum == 0:
        return {item: 0.5 for item in items}

    # 1) Roh-Heights
    heights = np.array([totals.get(i, 0) / total_sum for i in items], dtype=float)

    # 2) Mindesthöhe erzwingen
    heights = np.maximum(heights, min_height)

    # 3) Variable Gaps (kleine Nodes → größerer Abstand)
    h_max = heights.max()
    gaps = gap_base + gap_scale * (h_max - heights)

    # 4) Normalisieren
    total_span = heights.sum() + gaps.sum()
    heights /= total_span
    gaps    /= total_span

    # 5) Midpoints berechnen
    y_positions = {}
    y_current = 0.0
    for item, h, g in zip(items, heights, gaps):
        y_positions[item] = y_current + h / 2
        y_current += h + g

    return y_positions

source_y_positions = compute_y_positions(
    sources,
    source_totals,
    min_height=0.024,   # Sources meist weniger → etwas größer
    gap_base=0.02,
    gap_scale=0.020
)

target_y_positions = compute_y_positions(
    targets,
    target_totals,
    min_height=0.012,
    gap_base=0.002,
    gap_scale=0.020
)



# Erstelle Knoten-Arrays: Zuerst alle Sources, dann alle Targets
# Sources (links)
for source in sources:
    node_x.append(0.05)  # Links für Sources, mit etwas Abstand zum Rand
    node_y.append(source_y_positions.get(source, 0))
    if source == 'Added':
        node_colors.append("#843b77")
    else:
        node_colors.append("#2c36a0")
    # Label mit Gesamtwert - kürzere Formatierung
    total = source_totals.get(source, 0)
    label = source.replace('_other', ' (other)').replace('bicycle=designated', '| bicy=dsgn').replace('bicycle=yes', '| bicy=yes').replace('_', ' ')
    if len(label) > 30:
        label = label[:27] + '...'
    node_labels.append(f'{label}<br><b>{total:.1f} km</b>')

# Targets (rechts)
for target in targets:
    node_x.append(0.95)  # Rechts für Targets, mit etwas Abstand zum Rand
    node_y.append(target_y_positions.get(target, 0))
    node_colors.append("#4d9663")
    total = target_totals.get(target, 0)
    label = target.replace('bicycle=designated', '| bicy=dsgn').replace('bicycle=yes', '| bicy=yes').replace('_', ' ')
    if len(label) > 30:
        label = label[:27] + '...'
    node_labels.append(f'{label} <b>{total:.1f} km</b>')

# Erstelle das Sankey-Diagramm
fig = go.Figure(data=[go.Sankey(
    arrangement='perpendicular',  # Bessere Anordnung der Links
    node=dict(
        pad=25,  # Mehr Abstand zwischen Knoten
        thickness=20,  # Dickere Knoten für bessere Lesbarkeit
        line=dict(color="black", width=1),
        label=node_labels,
        color=node_colors,
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=values,
        color=link_colors,
        hovertemplate='<b>%{source.label}</b> → <b>%{target.label}</b><br>Wert: <b>%{value:.1f} km</b><extra></extra>'
    )
)])

print (node_y)

fig.update_layout(
    #title_text="<b>#missing-cw_mapillary-signs</b>, by 2025-12-01 <br><br> Infrastruktur-Änderungen: Sources → Targets <br>",
    title_text=(
    "<b>#missing-cw_mapillary-signs</b>, Stand: "+ DATE_SNAPSHOT +"<br><br>" 
    "Infrastruktur-Änderungen: Sources → Targets<br><br>"
    "<span style='font-size:11px;'>"
    "<b>Lesart:</b> Die Targets (rechts) zeigen Radinfrastruktur, "
    "die es zuvor in dieser Form noch nicht gab. <br>"
    "Verbindungen aus <i>Added</i> stehen für neu hinzugefügte Wege. "
    "Alle anderen Sources repräsentieren bestehende Wege, <br>"
    "die durch Tag-Anpassungen zur Radinfrasturkur ergänzt oder präzisiert wurden.<br>"
    "<br>"
    "<b>Hinweis:</b> „(other)“ auf der linken Seite bedeutet, "
    "dass zuvor keine eindeutig erkennbare Radinfrastruktur "
    "getaggt war – <br>häufig fehlten entweder "
    "<span style='font-family:monospace;font-size:11px;'>bicycle=yes/designated</span>"
    " auf separaten Wegen oder ein "
    "<span style='font-family:monospace;font-size:11px;'>cycleway</span>"
    "-Tag.</span>"

) ,
    font_size=11,
    height=1200,
    width=800,  # Etwas breiter für bessere Darstellung
    margin=dict(l=20, r=50, t=260, b=50),  # Mehr Rand für Labels
        title=dict(
        y=0.95,
        yanchor="top"
    ),
)

fig.show()


[np.float64(0.12008964932294829), np.float64(0.29643306802620883), np.float64(0.39170384577528855), np.float64(0.4621082734471145), np.float64(0.5158621964100638), np.float64(0.5680204288027227), np.float64(0.6167808517792137), np.float64(0.6576206218214176), np.float64(0.690931126939848), np.float64(0.7225781834708093), np.float64(0.7542252400017705), np.float64(0.7858722965327317), np.float64(0.8175193530636929), np.float64(0.8491664095946542), np.float64(0.8808134661256154), np.float64(0.9124605226565766), np.float64(0.9441075791875378), np.float64(0.9757546357184991), np.float64(0.3208575717961112), np.float64(0.6959609339347674), np.float64(0.7867718029283901), np.float64(0.8509084730264217), np.float64(0.9109646967481942), np.float64(0.9533359296587042), np.float64(0.9800785944710204)]


In [ ]:
#fig.write_html("sankey.html", include_plotlyjs="cdn")

In [29]:
###### GEoemetreiesn

In [31]:
def get_infrastructure_geometries_strict():
    transitions_df = get_infrastructure_transitions_strict_df().copy()
    transitions_df['geometry'] = transitions_df['geometry'].apply(
        lambda value: bytes(value) if isinstance(value, bytearray) else value
    )
    transitions_df['geometry'] = gpd.GeoSeries.from_wkb(transitions_df['geometry'])
    gdf = gpd.GeoDataFrame(transitions_df, geometry='geometry', crs='EPSG:4326')
    return gdf

In [32]:
gdf_strict = get_infrastructure_geometries_strict()
print(f"Strict rows: {len(gdf_strict)}")
gdf_strict.head()

Strict rows: 2620


,geometry,source,target,length_km,osm_id,osm_version,valid_from
0,"LINESTRING (7.00638 52.0764, 7.00644 52.07641,...",hw=tertiary_other,cw=track,0.119858,5054575,26,2025-09-07 18:30:27+02:00
1,"LINESTRING (6.60481 51.54364, 6.60474 51.54363...",hw=tertiary_other,cw=lane,0.364879,5121393,19,2025-10-14 20:59:03+02:00
2,"LINESTRING (6.15086 51.68366, 6.15135 51.68384)",hw=tertiary_other,cw=track,0.039346,10942638,6,2025-09-07 14:53:56+02:00
3,"LINESTRING (6.60027 51.5484, 6.59998 51.54827,...",hw=tertiary_other,cw=track,0.639190,21105527,19,2025-10-14 20:59:03+02:00
4,"LINESTRING (8.61471 50.22078, 8.6147 50.22072)",hw=residential_other,cw=lane,0.006528,23133583,18,2025-09-25 09:52:02+02:00


In [33]:
import folium

def plot_target_source_map(gdf, out_html, map_title):
    gdf_target = gdf.copy()
    gdf_source = gdf.copy()

    m = gdf_target.explore(
        column="target",
        name=f"{map_title} - Target",
        tiles="CartoDB Positron",
        attr="CartoDB Positron",
        style_kwds={"weight": 2, "opacity": 0.8},
        legend=True,
        legend_kwds={"caption": f"{map_title}: Target"},
        popup=True,
        tooltip=["source", "target", "length_km", "osm_id"]
    )

    gdf_source.explore(
        m=m,
        column="source",
        name=f"{map_title} - Source",
        style_kwds={"weight": 2, "opacity": 0.8},
        legend=True,
        legend_kwds={"caption": f"{map_title}: Source"},
        popup=True,
        tooltip=["source", "target", "length_km", "osm_id"],
        show=False
    )

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)
    print(f"✅ gespeichert: {out_html}")
    return m

gdf_strict = get_infrastructure_geometries_strict()
m_strict = plot_target_source_map(
    gdf_strict,
    "gdf_categories_strict.html",
    "Strict"
 )

✅ gespeichert: gdf_categories_strict.html


In [ ]:
################### EXPORT

In [38]:
gdf_strict

,geometry,source,target,length_km,osm_id,osm_version,valid_from
0,"LINESTRING (7.00638 52.0764, 7.00644 52.07641,...",hw=tertiary_other,cw=track,0.119858,5054575,26,2025-09-07 18:30:27+02:00
1,"LINESTRING (6.60481 51.54364, 6.60474 51.54363...",hw=tertiary_other,cw=lane,0.364879,5121393,19,2025-10-14 20:59:03+02:00
2,"LINESTRING (6.15086 51.68366, 6.15135 51.68384)",hw=tertiary_other,cw=track,0.039346,10942638,6,2025-09-07 14:53:56+02:00
3,"LINESTRING (6.60027 51.5484, 6.59998 51.54827,...",hw=tertiary_other,cw=track,0.639190,21105527,19,2025-10-14 20:59:03+02:00
4,"LINESTRING (8.61471 50.22078, 8.6147 50.22072)",hw=residential_other,cw=lane,0.006528,23133583,18,2025-09-25 09:52:02+02:00
...,...,...,...,...,...,...,...
2615,"LINESTRING (8.96796 54.26314, 8.9682 54.26281,...",hw=secondary_other,cw=track,0.718090,1032690180,4,2026-03-14 10:12:52+01:00
2616,"LINESTRING (8.96787 54.26343, 8.96783 54.26334...",hw=secondary_other,cw=track,0.034030,1032690182,5,2026-03-14 10:12:52+01:00
2617,"LINESTRING (9.1805 54.1714, 9.18122 54.17155, ...",hw=secondary_other,cw=track,0.181913,1385704010,3,2026-03-14 10:00:02+01:00
2618,"LINESTRING (9.17957 54.17071, 9.17998 54.17109...",hw=secondary_other,cw=track,0.099306,1391596170,3,2026-03-14 10:00:02+01:00


In [ ]:
## write necessary raw data

sankey_df

,source,target,value
0,Added,hw=path_bicycle=designated,208.6
1,hw=path_other,hw=path_bicycle=designated,82.7
2,hw=track_other,hw=path_bicycle=designated,35.7
3,hw=cycleway,hw=path_bicycle=designated,35.5
4,hw=track_other,hw=track_bicycle=designated,31.5
5,hw=secondary_other,cw=track,25.5
6,hw=footway_other,hw=path_bicycle=designated,24.4
7,hw=tertiary_other,cw=track,22.2
8,hw=footway_bicycle=yes,hw=path_bicycle=designated,17.7
9,Added,hw=footway_bicycle=yes,14.9


In [ ]:
from pathlib import Path

EXPORT_DIR = Path("viz/preprocessing/data")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SANKEY_JSON_PATH = EXPORT_DIR / "sankey.json"
TRANSITIONS_GEOJSON_PATH = EXPORT_DIR / "transitions_strict.geojson"
TRANSITIONS_GEOPARQUET_PATH = EXPORT_DIR / "transitions_strict.parquet"

# Sankey-Daten als frontend-freundliches JSON
sankey_export = sankey_df.copy()
sankey_export["value"] = sankey_export["value"].round(1)
sankey_export.to_json(
    SANKEY_JSON_PATH,
    orient="records",
    indent=2,
    force_ascii=False
)

# Geometriedaten fuer Karte / PMTiles-Vorstufe
gdf_export = gdf_strict.copy()

# Stabile ID fuer Frontend-Interaktion und spaetere Joins
gdf_export["feature_id"] = (
    gdf_export["osm_id"].astype(str) + "_" + gdf_export["osm_version"].astype(str)
)

# Zeitstempel fuer Web-Auslieferung robust als ISO-String
gdf_export["valid_from"] = (
    pd.to_datetime(gdf_export["valid_from"], utc=True)
    .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
)

# GeoJSON fuer einfache Entwicklung mit MapLibre
gdf_export.to_file(TRANSITIONS_GEOJSON_PATH, driver="GeoJSON")

# GeoParquet als effiziente Source-of-truth fuer weitere Verarbeitung
#gdf_export.to_parquet(TRANSITIONS_GEOPARQUET_PATH, index=False)

print(f"✅ Sankey JSON geschrieben: {SANKEY_JSON_PATH}")
print(f"✅ GeoJSON geschrieben: {TRANSITIONS_GEOJSON_PATH}")
#print(f"✅ GeoParquet geschrieben: {TRANSITIONS_GEOPARQUET_PATH}")
print()
print("Sankey rows:", len(sankey_export))
print("Transition features:", len(gdf_export))
print("Columns:", list(gdf_export.columns))

✅ Sankey JSON geschrieben: viz/preprocessing/data/sankey.json
✅ GeoJSON geschrieben: viz/preprocessing/data/transitions_strict.geojson
✅ GeoParquet geschrieben: viz/preprocessing/data/transitions_strict.parquet

Sankey rows: 46
Transition features: 2620
Columns: ['geometry', 'source', 'target', 'length_km', 'osm_id', 'osm_version', 'valid_from', 'feature_id']
